# Resume Parser — Evaluation Stage
This notebook evaluates **job-ranking quality** using a small, explicitly labeled dataset of resume/job-description pairs.

**Important:** The included examples are synthetic starter examples. Replace or expand them with manually reviewed pairs before reporting results. Metrics from a tiny synthetic set are only a pipeline check, not evidence of real-world performance.

## 1. Install and import dependencies
This stage uses the same semantic model as the upgraded notebook. If it is already installed in your environment, the install command should finish quickly.

In [ ]:
%pip install -q sentence-transformers scikit-learn pandas numpy

In [ ]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 160)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)
print("Loaded:", MODEL_NAME)

## 2. Create a small labeled evaluation set
Each row represents a resume paired with a job. `relevance` is a human-assigned label:
- `2` = highly relevant
- `1` = partially relevant
- `0` = not relevant

These are illustrative profiles, not real candidate data. Edit the profiles and labels to reflect human judgment. For a meaningful evaluation, aim for many resumes and several jobs per resume.

In [ ]:
evaluation_data = [
    {
        "resume_id": "R1", "role": "ML Engineer", "relevance": 2,
        "resume_text": "Computer science student. Python, machine learning, pandas, NumPy, scikit-learn, model evaluation, feature engineering, PyTorch, NLP, Git. Built a classification model and deployed a small FastAPI service."
    },
    {
        "resume_id": "R1", "role": "Data Analyst", "relevance": 1,
        "resume_text": "Computer science student. Python, machine learning, pandas, NumPy, scikit-learn, model evaluation, feature engineering, PyTorch, NLP, Git. Built a classification model and deployed a small FastAPI service."
    },
    {
        "resume_id": "R1", "role": "Backend Python Developer", "relevance": 1,
        "resume_text": "Computer science student. Python, machine learning, pandas, NumPy, scikit-learn, model evaluation, feature engineering, PyTorch, NLP, Git. Built a classification model and deployed a small FastAPI service."
    },
    {
        "resume_id": "R2", "role": "Data Analyst", "relevance": 2,
        "resume_text": "Entry-level data analyst with SQL, Excel, pandas, statistics, data cleaning, dashboards, data visualization, Power BI, reporting, and stakeholder communication."
    },
    {
        "resume_id": "R2", "role": "ML Engineer", "relevance": 1,
        "resume_text": "Entry-level data analyst with SQL, Excel, pandas, statistics, data cleaning, dashboards, data visualization, Power BI, reporting, and stakeholder communication."
    },
    {
        "resume_id": "R2", "role": "Backend Python Developer", "relevance": 0,
        "resume_text": "Entry-level data analyst with SQL, Excel, pandas, statistics, data cleaning, dashboards, data visualization, Power BI, reporting, and stakeholder communication."
    },
    {
        "resume_id": "R3", "role": "Backend Python Developer", "relevance": 2,
        "resume_text": "Python backend developer. FastAPI, Flask, REST APIs, PostgreSQL, SQL, authentication, unit testing, Docker, Git, CI/CD, and building maintainable services."
    },
    {
        "resume_id": "R3", "role": "ML Engineer", "relevance": 0,
        "resume_text": "Python backend developer. FastAPI, Flask, REST APIs, PostgreSQL, SQL, authentication, unit testing, Docker, Git, CI/CD, and building maintainable services."
    },
    {
        "resume_id": "R3", "role": "Data Analyst", "relevance": 1,
        "resume_text": "Python backend developer. FastAPI, Flask, REST APIs, PostgreSQL, SQL, authentication, unit testing, Docker, Git, CI/CD, and building maintainable services."
    },
    {
        "resume_id": "R4", "role": "ML Engineer", "relevance": 2,
        "resume_text": "AI engineer with deep learning, PyTorch, TensorFlow, computer vision, transformers, NLP, experiment tracking, evaluation metrics, and model deployment."
    },
    {
        "resume_id": "R4", "role": "Data Analyst", "relevance": 0,
        "resume_text": "AI engineer with deep learning, PyTorch, TensorFlow, computer vision, transformers, NLP, experiment tracking, evaluation metrics, and model deployment."
    },
    {
        "resume_id": "R4", "role": "Backend Python Developer", "relevance": 1,
        "resume_text": "AI engineer with deep learning, PyTorch, TensorFlow, computer vision, transformers, NLP, experiment tracking, evaluation metrics, and model deployment."
    },
]
job_descriptions = {
    "ML Engineer": "Build, train, evaluate, and deploy machine learning models. Python, scikit-learn, PyTorch, feature engineering, NLP, deep learning, model evaluation, APIs, and data pipelines.",
    "Data Analyst": "Analyze business data using SQL, Python, pandas, Excel, statistics, dashboards, Power BI, data visualization, reporting, and stakeholder communication.",
    "Backend Python Developer": "Develop backend services with Python, FastAPI or Flask, REST APIs, SQL databases, PostgreSQL, authentication, automated testing, Docker, and Git."
}

eval_df = pd.DataFrame(evaluation_data)
eval_df[["resume_id", "role", "relevance"]]

## 3. Build a ranking for each resume
We will rank the job descriptions using semantic similarity and TF-IDF similarity. A model score is not a ground-truth label; the human relevance labels above are the reference for metric calculation.

In [ ]:
def tfidf_scores(resume_text, jobs):
    texts = [resume_text] + [jobs[role] for role in jobs]
    matrix = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)).fit_transform(texts)
    return cosine_similarity(matrix[0:1], matrix[1:]).ravel()

def semantic_scores(resume_text, jobs, model):
    texts = [resume_text] + [jobs[role] for role in jobs]
    embeddings = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return embeddings[1:] @ embeddings[0]

def rank_all_resumes(df, jobs, model):
    rows = []
    roles = list(jobs.keys())
    for resume_id, group in df.groupby("resume_id"):
        resume_text = group["resume_text"].iloc[0]
        sem = semantic_scores(resume_text, jobs, model)
        lex = tfidf_scores(resume_text, jobs)
        # Blend two ranking signals. Keep weights fixed during initial evaluation.
        blended = 0.7 * sem + 0.3 * lex
        human = group.set_index("role")["relevance"].to_dict()
        for i, role in enumerate(roles):
            rows.append({
                "resume_id": resume_id,
                "role": role,
                "semantic_cosine": float(sem[i]),
                "tfidf_cosine": float(lex[i]),
                "blended_score": float(blended[i]),
                "relevance": int(human.get(role, 0))
            })
    return pd.DataFrame(rows)

rankings = rank_all_resumes(eval_df, job_descriptions, model)
rankings.sort_values(["resume_id", "blended_score"], ascending=[True, False])

## 4. Ranking metrics: Precision@K, Recall@K, and nDCG
For these calculations, relevance `1` or `2` counts as relevant for Precision@K and Recall@K. nDCG uses the graded labels 0, 1, and 2. We also report MRR (mean reciprocal rank) to show where the first relevant role appears.

In [ ]:
def precision_at_k(relevances, k=1):
    rel = np.asarray(relevances[:k]) > 0
    return float(rel.sum() / k)

def recall_at_k(relevances, k=1):
    rel_all = np.asarray(relevances) > 0
    if rel_all.sum() == 0:
        return np.nan
    return float((np.asarray(relevances[:k]) > 0).sum() / rel_all.sum())

def dcg_at_k(relevances, k=3):
    rel = np.asarray(relevances[:k], dtype=float)
    if len(rel) == 0:
        return 0.0
    gains = (2 ** rel - 1)
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))

def ndcg_at_k(relevances, k=3):
    actual = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return float(actual / ideal) if ideal > 0 else np.nan

def reciprocal_rank(relevances):
    for rank, relevance in enumerate(relevances, start=1):
        if relevance > 0:
            return 1.0 / rank
    return 0.0

def evaluate_rankings(ranking_df, k=1):
    rows = []
    for resume_id, group in ranking_df.groupby("resume_id"):
        ordered = group.sort_values("blended_score", ascending=False)
        rels = ordered["relevance"].tolist()
        rows.append({
            "resume_id": resume_id,
            f"Precision@{k}": precision_at_k(rels, k),
            f"Recall@{k}": recall_at_k(rels, k),
            "nDCG@3": ndcg_at_k(rels, 3),
            "Reciprocal rank": reciprocal_rank(rels),
            "Top-ranked role": ordered.iloc[0]["role"],
            "Top-ranked relevance": int(ordered.iloc[0]["relevance"]),
        })
    per_resume = pd.DataFrame(rows)
    metric_cols = [f"Precision@{k}", f"Recall@{k}", "nDCG@3", "Reciprocal rank"]
    aggregate = per_resume[metric_cols].mean(numeric_only=True).to_frame("Mean").T
    return per_resume, aggregate

per_resume_metrics, aggregate_metrics = evaluate_rankings(rankings, k=1)
display(per_resume_metrics)
display(aggregate_metrics)

## 5. Compare semantic-only, TF-IDF-only, and blended ranking
This helps check whether blending signals helps on the labeled examples. Do not tune weights repeatedly against a tiny test set and then report the same result as an unbiased evaluation.

In [ ]:
def evaluate_scoring_column(ranking_df, score_col):
    rows = []
    for resume_id, group in ranking_df.groupby("resume_id"):
        ordered = group.sort_values(score_col, ascending=False)
        rels = ordered["relevance"].tolist()
        rows.append({
            "resume_id": resume_id,
            "Precision@1": precision_at_k(rels, 1),
            "Recall@1": recall_at_k(rels, 1),
            "nDCG@3": ndcg_at_k(rels, 3),
            "MRR": reciprocal_rank(rels),
            "Top role": ordered.iloc[0]["role"]
        })
    per = pd.DataFrame(rows)
    return per[["Precision@1", "Recall@1", "nDCG@3", "MRR"]].mean().rename(score_col)

comparison = pd.concat([
    evaluate_scoring_column(rankings, "semantic_cosine"),
    evaluate_scoring_column(rankings, "tfidf_cosine"),
    evaluate_scoring_column(rankings, "blended_score"),
], axis=1).T
comparison

## 6. Inspect errors
Look for resumes whose top-ranked job is labeled irrelevant or only partially relevant. These examples are useful for improving the skill vocabulary, job descriptions, and matching logic.

In [ ]:
rankings["rank"] = rankings.groupby("resume_id")["blended_score"].rank(method="first", ascending=False).astype(int)
top_predictions = rankings[rankings["rank"] == 1].sort_values("resume_id")
top_predictions[["resume_id", "role", "blended_score", "relevance"]]

## 7. Export evaluation results

In [ ]:
rankings.to_csv("resume_job_ranking_evaluation.csv", index=False)
per_resume_metrics.to_csv("resume_ranking_metrics_by_resume.csv", index=False)
comparison.to_csv("resume_ranking_metric_comparison.csv")
print("Saved:")
print("- resume_job_ranking_evaluation.csv")
print("- resume_ranking_metrics_by_resume.csv")
print("- resume_ranking_metric_comparison.csv")

## 8. How to make this evaluation credible
1. Collect consented or public resumes and real job descriptions.
2. Ask a human reviewer to label each resume–job pair before looking at model scores.
3. Include hard negatives: jobs with overlapping vocabulary but different responsibilities.
4. Use separate development and test sets. Choose weights on development data, then evaluate once on held-out test data.
5. For skill extraction, manually annotate skills and calculate precision, recall, and F1 separately from ranking metrics.
6. Report the number of resumes, jobs, labeled pairs, and how labels were assigned.

**Do not claim production-grade accuracy from this tiny synthetic dataset.** It verifies that the evaluation code runs and illustrates the metrics; it does not establish real-world performance.